# NLP Project I: Structured Text Classification & Information Extraction

This notebook evaluates a local open-source LLM (`qwen2.5:3b` via Ollama) on **sentiment classification** and **named person entity extraction** from movie reviews (`stanfordnlp/imdb`).

We compare two decoding settings across 100 gold-annotated samples:
1. **Prompt-Only:** JSON format requested purely via instructions (no `format` parameter).
2. **Schema-Constrained:** Strict JSON Schema enforced via Ollama's `format` parameter.

In [1]:
# 1. Configuration and Model Verification
import json
import os
import re
import time

import ollama
import pandas as pd
from sklearn.metrics import accuracy_score, f1_score

# Fixed experiment parameters for reproducibility
MODEL_NAME = "qwen2.5:3b"
SEED = 42
TEMPERATURE = 0.0
NUM_CTX = 2048
RESULTS_DIR = "results"

os.makedirs(RESULTS_DIR, exist_ok=True)

# Inspect local model metadata from Ollama
model_info = ollama.show(MODEL_NAME)
details = model_info.details
print("Ollama Python client ready")
print(f"Model:        {MODEL_NAME}")
print(f"Family:       {details.family}")
print(f"Parameters:   {details.parameter_size}")
print(f"Quantization: {details.quantization_level}")
print(f"Context len:  {model_info.modelinfo.get('qwen2.context_length')}")
print(f"Seed: {SEED} | Temperature: {TEMPERATURE} | num_ctx: {NUM_CTX}")

Ollama Python client ready
Model:        qwen2.5:3b
Family:       qwen2
Parameters:   3.1B
Quantization: Q4_K_M
Context len:  32768
Seed: 42 | Temperature: 0.0 | num_ctx: 2048


## 2. Load Dataset and Gold Annotations

- Full local source copy: `data/stanfordnlp_imdb/imdb_test.parquet` (25,000 test reviews).
- Annotated evaluation subset: `data/dataset_100.csv` (50 positive, 50 negative reviews with manually verified gold person entities).

In [2]:
# 2. Load source dataset stats and the 100-sample annotated subset
full_imdb = pd.read_parquet("data/stanfordnlp_imdb/imdb_test.parquet")
print(f"Full Stanford IMDB test split rows: {len(full_imdb)}")

df = pd.read_csv("data/dataset_100.csv")
df["gold_entities"] = df["gold_entities"].apply(json.loads)

print(f"Evaluation subset rows: {len(df)}")
print("Sentiment distribution:", df["gold_sentiment"].value_counts().to_dict())
print(f"Total gold entity mentions: {df['gold_entities'].apply(len).sum()}")

df[["id", "gold_sentiment", "gold_entities", "text"]].head(5)

Full Stanford IMDB test split rows: 25000
Evaluation subset rows: 100
Sentiment distribution: {'positive': 50, 'negative': 50}
Total gold entity mentions: 127


,id,gold_sentiment,gold_entities,text
0,1,positive,[Aunty Lou],"A delightful story about two evacuees, has bee..."
1,2,positive,"[Shawn Phelan, Derek, Yogurt]","Yes, it's not a great cinematic achievement, b..."
2,3,positive,"[Gheorghe Muresan, Billy Crystal]",I just wanted to say that. I love Gheorghe Mur...
3,4,positive,"[Renoir, Jean Gabin]","Renoir's tale of Paris,the Cancan,a washer-gir..."
4,5,negative,[Alexandra Paul],I saw this Film one midnight and I can say tha...


## 3. Prompt Template and JSON Schema

Both experimental settings use the exact same system prompt and user prompt. The only difference is whether `RESPONSE_SCHEMA` is passed to Ollama's `format` parameter.

In [3]:
# 3. Prompt definition and JSON Schema
SYSTEM_PROMPT = (
    "You are an NLP information extraction system. "
    "Analyze the given movie review and respond ONLY with a valid JSON object "
    "with these exact keys:\n"
    '- "sentiment": either "positive" or "negative"\n'
    '- "sentiment_score": a float from 0.0 (very negative) to 1.0 (very positive)\n'
    '- "entities": a list of named people explicitly mentioned in the review '
    "(actors, directors, writers, or character names), copied using their exact "
    "names from the text, or [] if no named person is mentioned. "
    "Do not include movie titles, studios, or locations."
)


def build_user_prompt(review_text: str) -> str:
    """Format the movie review into a user prompt."""
    return f"Review:\n{review_text}"


# JSON Schema used exclusively in the schema-constrained setting
RESPONSE_SCHEMA = {
    "type": "object",
    "properties": {
        "sentiment": {"type": "string", "enum": ["positive", "negative"]},
        "sentiment_score": {"type": "number", "minimum": 0.0, "maximum": 1.0},
        "entities": {"type": "array", "items": {"type": "string"}},
    },
    "required": ["sentiment", "sentiment_score", "entities"],
    "additionalProperties": False,
}

## 4. Model Inference, JSON Cleaning, and Schema Validation

In the **prompt-only** setting, models may wrap JSON inside Markdown fences (```` ```json ````) or prepend conversational text. We track both:
- **Raw parsing:** `json.loads()` directly on the raw string.
- **Cleaned parsing:** `json.loads()` after removing Markdown fences and extracting the `{...}` block, wrapped in `try-except`.

In [4]:
# 4. Model inference, cleaning, safe JSON parsing, and schema checks

def call_model(review_text: str, use_schema: bool = False, temperature: float = TEMPERATURE):
    """Call local Ollama model and return (raw_output, elapsed_seconds)."""
    kwargs = {
        "model": MODEL_NAME,
        "messages": [
            {"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user", "content": build_user_prompt(review_text)},
        ],
        "options": {
            "temperature": temperature,
            "seed": SEED,
            "num_ctx": NUM_CTX,
        },
    }
    # Pass JSON Schema only in schema-constrained mode
    if use_schema:
        kwargs["format"] = RESPONSE_SCHEMA

    start_time = time.time()
    response = ollama.chat(**kwargs)
    elapsed = time.time() - start_time
    return response["message"]["content"], elapsed


def clean_json_text(raw_text: str) -> str:
    """Remove Markdown code fences and extract the JSON object substring."""
    text = raw_text.strip()
    # Strip ```json ... ``` or ``` ... ``` wrappers if present
    text = re.sub(r"^```(?:json)?\s*", "", text, flags=re.IGNORECASE)
    text = re.sub(r"\s*```$", "", text)

    # Extract substring from the first '{' to the last '}'
    start_idx = text.find("{")
    end_idx = text.rfind("}")
    if start_idx != -1 and end_idx != -1 and end_idx > start_idx:
        return text[start_idx : end_idx + 1]
    return text


def parse_json_safe(text: str):
    """Safely parse JSON text and return None on failure."""
    try:
        return json.loads(text)
    except (json.JSONDecodeError, TypeError):
        return None


def is_valid_schema(data) -> bool:
    """Check that parsed JSON has all required fields with valid types."""
    if not isinstance(data, dict):
        return False

    required_keys = {"sentiment", "sentiment_score", "entities"}
    if not required_keys.issubset(data.keys()):
        return False

    if data["sentiment"] not in ("positive", "negative"):
        return False

    score = data["sentiment_score"]
    if isinstance(score, bool) or not isinstance(score, (int, float)):
        return False
    if not (0.0 <= float(score) <= 1.0):
        return False

    entities = data["entities"]
    if not isinstance(entities, list) or not all(isinstance(e, str) for e in entities):
        return False

    return True

## 5. Evaluation Metrics

We evaluate four aspects on every experimental setting:
1. **Raw JSON Parsing Success Rate** (before cleaning)
2. **Cleaned JSON Parsing Success Rate** (after cleaning)
3. **Schema Validity Rate** (required keys and types)
4. **Task Performance:** Sentiment Accuracy & Macro F1, plus Entity Extraction Micro Precision / Recall / F1 using normalized exact matching (lowercase + stripped punctuation).

In [5]:
# 5. Evaluation helper functions

def normalize_entity(name: str) -> str:
    """Normalize an entity string for case-insensitive exact matching."""
    return name.strip().lower().strip(".,!?;:\"'()")


def evaluate_predictions(dataframe: pd.DataFrame, raw_outputs: list, latencies: list):
    """Compute format validity and task accuracy/F1 metrics."""
    n = len(dataframe)
    raw_ok_count = 0
    clean_ok_count = 0
    schema_ok_count = 0

    y_true = dataframe["gold_sentiment"].tolist()
    y_pred = []

    tp = fp = fn = 0
    records = []

    for idx, (_, row) in enumerate(dataframe.iterrows()):
        raw = raw_outputs[idx]

        # Step 1: Parse before any cleaning
        raw_parsed = parse_json_safe(raw)
        raw_ok = raw_parsed is not None
        if raw_ok:
            raw_ok_count += 1

        # Step 2: Parse after cleaning
        cleaned = clean_json_text(raw)
        parsed = parse_json_safe(cleaned)
        clean_ok = parsed is not None
        if clean_ok:
            clean_ok_count += 1

        # Step 3: Validate schema fields and types
        schema_ok = clean_ok and is_valid_schema(parsed)
        if schema_ok:
            schema_ok_count += 1
            pred_sentiment = parsed["sentiment"]
            pred_score = float(parsed["sentiment_score"])
            pred_entities = [e for e in parsed["entities"] if e.strip()]
        else:
            pred_sentiment = "invalid"
            pred_score = None
            pred_entities = []

        y_pred.append(pred_sentiment)

        # Step 4: Entity set comparison (micro TP / FP / FN)
        gold_set = {normalize_entity(e) for e in row["gold_entities"] if normalize_entity(e)}
        pred_set = {normalize_entity(e) for e in pred_entities if normalize_entity(e)}

        tp += len(gold_set & pred_set)
        fp += len(pred_set - gold_set)
        fn += len(gold_set - pred_set)

        records.append({
            "id": int(row["id"]),
            "gold_sentiment": row["gold_sentiment"],
            "pred_sentiment": pred_sentiment,
            "pred_score": pred_score,
            "gold_entities": row["gold_entities"],
            "pred_entities": pred_entities,
            "raw_json_ok": raw_ok,
            "cleaned_json_ok": clean_ok,
            "schema_ok": schema_ok,
            "latency_sec": round(latencies[idx], 3),
            "raw_output": raw,
        })

    sent_acc = accuracy_score(y_true, y_pred)
    sent_f1 = f1_score(y_true, y_pred, labels=["negative", "positive"], average="macro", zero_division=0)

    ent_precision = tp / (tp + fp) if (tp + fp) > 0 else 0.0
    ent_recall = tp / (tp + fn) if (tp + fn) > 0 else 0.0
    ent_f1 = (2 * ent_precision * ent_recall / (ent_precision + ent_recall)) if (ent_precision + ent_recall) > 0 else 0.0

    summary = {
        "samples": n,
        "raw_json_parse_rate": round(raw_ok_count / n * 100, 2),
        "cleaned_json_parse_rate": round(clean_ok_count / n * 100, 2),
        "schema_validity_rate": round(schema_ok_count / n * 100, 2),
        "sentiment_accuracy": round(sent_acc * 100, 2),
        "sentiment_macro_f1": round(sent_f1 * 100, 2),
        "entity_precision": round(ent_precision * 100, 2),
        "entity_recall": round(ent_recall * 100, 2),
        "entity_f1": round(ent_f1 * 100, 2),
        "avg_latency_sec": round(sum(latencies) / n, 3),
    }
    return summary, records

## 6. 100-Sample Experiment: Prompt-Only vs Schema-Constrained

We run all 100 reviews under both settings with identical model, seed, and temperature, then save the predictions and summary metrics to `results/`.

In [6]:
# 6. Run 100-sample evaluation for both settings
def run_experiment(dataframe: pd.DataFrame, use_schema: bool):
    """Run inference over all rows in the dataset."""
    raw_outputs = []
    latencies = []
    mode_label = "Schema-Constrained" if use_schema else "Prompt-Only"
    t_start = time.time()

    for idx, (_, row) in enumerate(dataframe.iterrows()):
        raw, dt = call_model(row["text"], use_schema=use_schema, temperature=TEMPERATURE)
        raw_outputs.append(raw)
        latencies.append(dt)
        if (idx + 1) % 25 == 0:
            print(f"[{mode_label}] Completed {idx + 1}/{len(dataframe)} samples")

    print(f"[{mode_label}] Finished in {time.time() - t_start:.1f}s")
    return raw_outputs, latencies


raw_prompt_only, lat_prompt_only = run_experiment(df, use_schema=False)
raw_schema, lat_schema = run_experiment(df, use_schema=True)

summary_prompt, records_prompt = evaluate_predictions(df, raw_prompt_only, lat_prompt_only)
summary_schema, records_schema = evaluate_predictions(df, raw_schema, lat_schema)

summary_prompt["setting"] = "Prompt-Only"
summary_schema["setting"] = "Schema-Constrained"

results_df = pd.DataFrame([summary_prompt, summary_schema]).set_index("setting")
results_df.to_csv(os.path.join(RESULTS_DIR, "metrics_summary.csv"))

with open(os.path.join(RESULTS_DIR, "predictions_prompt_only.json"), "w", encoding="utf-8") as f:
    json.dump(records_prompt, f, indent=2, ensure_ascii=False)

with open(os.path.join(RESULTS_DIR, "predictions_schema.json"), "w", encoding="utf-8") as f:
    json.dump(records_schema, f, indent=2, ensure_ascii=False)

results_df

[Prompt-Only] Completed 25/100 samples


[Prompt-Only] Completed 50/100 samples


[Prompt-Only] Completed 75/100 samples


[Prompt-Only] Completed 100/100 samples
[Prompt-Only] Finished in 125.1s


[Schema-Constrained] Completed 25/100 samples


[Schema-Constrained] Completed 50/100 samples


[Schema-Constrained] Completed 75/100 samples


[Schema-Constrained] Completed 100/100 samples
[Schema-Constrained] Finished in 166.2s


,samples,raw_json_parse_rate,cleaned_json_parse_rate,schema_validity_rate,sentiment_accuracy,sentiment_macro_f1,entity_precision,entity_recall,entity_f1,avg_latency_sec
setting,,,,,,,,,,
Prompt-Only,100,99.0,99.0,99.0,93.0,93.45,67.48,86.61,75.86,1.250
Schema-Constrained,100,100.0,100.0,100.0,94.0,94.00,67.07,86.61,75.60,1.662


In [7]:
# Inspect differences and any raw parsing / prediction mismatches between settings
raw_failures = [r for r in records_prompt if not r["raw_json_ok"]]
schema_failures = [r for r in records_prompt if not r["schema_ok"]]
sentiment_diffs = [
    (p, s) for p, s in zip(records_prompt, records_schema)
    if p["pred_sentiment"] != s["pred_sentiment"]
]
entity_diffs = [
    (p, s) for p, s in zip(records_prompt, records_schema)
    if {normalize_entity(e) for e in p["pred_entities"]} != {normalize_entity(e) for e in s["pred_entities"]}
]

print(f"Prompt-Only raw JSON parse failures: {len(raw_failures)}")
print(f"Prompt-Only schema validity failures: {len(schema_failures)}")
print(f"Sentiment prediction differences between settings: {len(sentiment_diffs)}")
print(f"Entity prediction differences between settings: {len(entity_diffs)}")

for item in raw_failures[:3]:
    print("\n--- Raw JSON Parse Failure Example ---")
    print(f"Sample ID: {item['id']} | Cleaned OK: {item['cleaned_json_ok']} | Schema OK: {item['schema_ok']}")
    print("Raw output:", repr(item["raw_output"]))

if not raw_failures and (sentiment_diffs or entity_diffs):
    print("\n--- Example Prediction Difference (Prompt-Only vs Schema-Constrained) ---")
    p, s = (sentiment_diffs or entity_diffs)[0]
    print(f"Sample ID: {p['id']} | Gold: sent={p['gold_sentiment']}, ent={p['gold_entities']}")
    print(f"Prompt-Only:       sent={p['pred_sentiment']}, ent={p['pred_entities']}")
    print(f"Schema-Constrained: sent={s['pred_sentiment']}, ent={s['pred_entities']}")

Prompt-Only raw JSON parse failures: 1
Prompt-Only schema validity failures: 1
Sentiment prediction differences between settings: 1
Entity prediction differences between settings: 1

--- Raw JSON Parse Failure Example ---
Sample ID: 58 | Cleaned OK: False | Schema OK: False
Raw output: '{\n\n{\n  "sentiment": "positive",\n  "sentiment_score": 0.85,\n  "entities": [\n    "Jonathan Silver"\n  ]\n}'


## 7. Live Tests

Interactive cells to inspect model behavior directly on individual samples:
- **7.1:** Prompt-Only vs Schema-Constrained on the same dataset reviews
- **7.2:** Temperature comparison (`0.0` vs `1.0`) across repeated runs
- **7.3:** Five unseen reviews outside the dataset (including one sarcastic review)

In [8]:
# 7.1 Live Test: Prompt-Only vs Schema-Constrained on 2 dataset samples
for row_idx in [0, 4]:
    row = df.iloc[row_idx]
    print("=" * 70)
    print(f"Sample ID {row['id']} | Gold: {row['gold_sentiment']} | Entities: {row['gold_entities']}")
    print("Text:", row["text"])

    raw_p, dt_p = call_model(row["text"], use_schema=False, temperature=0.0)
    raw_s, dt_s = call_model(row["text"], use_schema=True, temperature=0.0)

    print(f"\n[Prompt-Only | {dt_p:.2f}s] Raw parse OK={parse_json_safe(raw_p) is not None}")
    print(raw_p)
    print(f"\n[Schema-Constrained | {dt_s:.2f}s] Raw parse OK={parse_json_safe(raw_s) is not None}")
    print(raw_s)

Sample ID 1 | Gold: positive | Entities: ['Aunty Lou']
Text: A delightful story about two evacuees, has been turned into a nice little film, by the BBC. Most children who like a good story will enjoy this. The characters are played really well by a very good cast. Not sure whether our American friends will appreciate it, but they do get a mention, as Aunty Lou runs off with a gorgeous American soldier.



[Prompt-Only | 1.80s] Raw parse OK=True
{
    "sentiment": "positive",
    "sentiment_score": 0.85,
    "entities": [
        "Aunty Lou",
        "American soldier"
    ]
}

[Schema-Constrained | 1.48s] Raw parse OK=True
{
    "sentiment": "positive",
    "sentiment_score": 0.85,
    "entities": [
        "Aunty Lou",
        "American soldier"
    ]
}
Sample ID 5 | Gold: negative | Entities: ['Alexandra Paul']
Text: I saw this Film one midnight and I can say that it worse than other horror film about a Haunted House.Alexandra Paul is not one of the best actress but she can do the role better,The little girl get worse this is a example about a Bad actress,she has not got future in the great world of films. SENTENCE FOR HOUSE OF THE DAMNED:BAD



[Prompt-Only | 1.81s] Raw parse OK=True
{
    "sentiment": "negative",
    "sentiment_score": 0.4,
    "entities": [
        "Alexandra Paul",
        "The little girl"
    ]
}

[Schema-Constrained | 1.47s] Raw parse OK=True
{
    "sentiment": "negative",
    "sentiment_score": 0.4,
    "entities": [
        "Alexandra Paul",
        "The little girl"
    ]
}


In [9]:
# 7.2 Live Test: Temperature 0.0 vs 1.0 (two runs each on the same samples)
for row_idx in [1, 5]:
    row = df.iloc[row_idx]
    print("=" * 70)
    print(f"Sample ID {row['id']} | Gold: {row['gold_sentiment']} | Entities: {row['gold_entities']}")

    for temp in [0.0, 1.0]:
        for run_num in [1, 2]:
            raw_t, dt_t = call_model(row["text"], use_schema=False, temperature=temp)
            parsed_t = parse_json_safe(clean_json_text(raw_t))
            raw_ok = parse_json_safe(raw_t) is not None
            print(f"  temp={temp} run={run_num} ({dt_t:.2f}s) raw_ok={raw_ok} -> {parsed_t}")

Sample ID 2 | Gold: positive | Entities: ['Shawn Phelan', 'Derek', 'Yogurt']


  temp=0.0 run=1 (2.07s) raw_ok=True -> {'sentiment': 'positive', 'sentiment_score': 0.85, 'entities': ['Shawn Phelan', 'Derek', 'Yogurt']}


  temp=0.0 run=2 (1.72s) raw_ok=True -> {'sentiment': 'positive', 'sentiment_score': 0.85, 'entities': ['Shawn Phelan', 'Derek', 'Yogurt']}


  temp=1.0 run=1 (1.74s) raw_ok=True -> {'sentiment': 'positive', 'sentiment_score': 0.85, 'entities': ['Shawn Phelan', 'Derek', 'Yogurt']}


  temp=1.0 run=2 (1.75s) raw_ok=True -> {'sentiment': 'positive', 'sentiment_score': 0.85, 'entities': ['Shawn Phelan', 'Derek', 'Yogurt']}
Sample ID 6 | Gold: negative | Entities: ['Van Damme']


  temp=0.0 run=1 (1.62s) raw_ok=True -> {'sentiment': 'positive', 'sentiment_score': 0.7, 'entities': ['Van Damme']}


  temp=0.0 run=2 (1.25s) raw_ok=True -> {'sentiment': 'positive', 'sentiment_score': 0.7, 'entities': ['Van Damme']}


  temp=1.0 run=1 (1.26s) raw_ok=True -> {'sentiment': 'positive', 'sentiment_score': 0.7, 'entities': ['Van Damme']}


  temp=1.0 run=2 (1.27s) raw_ok=True -> {'sentiment': 'positive', 'sentiment_score': 0.7, 'entities': ['Van Damme']}


In [10]:
# 7.3 Live Test: 5 Unseen Reviews (not in dataset, #5 is intentionally sarcastic)
unseen_reviews = [
    {
        "note": "Clear positive with director and actor",
        "expected": "positive | ['Christopher Nolan', 'Cillian Murphy']",
        "text": "Oppenheimer is an unforgettable masterpiece. Christopher Nolan keeps the tension high and Cillian Murphy gives a career-best performance.",
    },
    {
        "note": "Clear negative with actor",
        "expected": "negative | ['Adam Sandler']",
        "text": "This comedy is painful to watch. The jokes fall flat in every scene and even Adam Sandler looks bored throughout the entire movie.",
    },
    {
        "note": "Clear positive with actor and movie title (title excluded)",
        "expected": "positive | ['Tom Hanks']",
        "text": "Forrest Gump is a timeless classic. Tom Hanks brings warmth and humor to every single scene from start to finish.",
    },
    {
        "note": "Positive with multiple crew/cast names",
        "expected": "positive | ['Denis Villeneuve', 'Rebecca Ferguson', 'Timothée Chalamet']",
        "text": "Denis Villeneuve crafted a visually stunning sci-fi epic, carried by strong performances from Rebecca Ferguson and Timothée Chalamet.",
    },
    {
        "note": "Sarcastic negative (expected to fool the model into positive)",
        "expected": "negative | ['Michael Bay']",
        "text": "This movie really cured my insomnia! Thank you, director Michael Bay, for making the longest ten-minute story feel like a peaceful five-hour nap.",
    },
]

for i, item in enumerate(unseen_reviews, start=1):
    raw_u, dt_u = call_model(item["text"], use_schema=True, temperature=0.0)
    parsed_u = parse_json_safe(raw_u)
    print(f"--- Unseen #{i}: {item['note']} ---")
    print("Text:    ", item["text"])
    print("Expected:", item["expected"])
    print(f"Model ({dt_u:.2f}s):", parsed_u)

--- Unseen #1: Clear positive with director and actor ---
Text:     Oppenheimer is an unforgettable masterpiece. Christopher Nolan keeps the tension high and Cillian Murphy gives a career-best performance.
Expected: positive | ['Christopher Nolan', 'Cillian Murphy']
Model (1.60s): {'sentiment': 'positive', 'sentiment_score': 0.85, 'entities': ['Christopher Nolan', 'Cillian Murphy']}


--- Unseen #2: Clear negative with actor ---
Text:     This comedy is painful to watch. The jokes fall flat in every scene and even Adam Sandler looks bored throughout the entire movie.
Expected: negative | ['Adam Sandler']
Model (1.38s): {'sentiment': 'negative', 'sentiment_score': 0.1, 'entities': ['Adam Sandler']}


--- Unseen #3: Clear positive with actor and movie title (title excluded) ---
Text:     Forrest Gump is a timeless classic. Tom Hanks brings warmth and humor to every single scene from start to finish.
Expected: positive | ['Tom Hanks']
Model (1.41s): {'sentiment': 'positive', 'sentiment_score': 0.85, 'entities': ['Tom Hanks']}


--- Unseen #4: Positive with multiple crew/cast names ---
Text:     Denis Villeneuve crafted a visually stunning sci-fi epic, carried by strong performances from Rebecca Ferguson and Timothée Chalamet.
Expected: positive | ['Denis Villeneuve', 'Rebecca Ferguson', 'Timothée Chalamet']
Model (2.14s): {'sentiment': 'positive', 'sentiment_score': 0.85, 'entities': ['Denis Villeneuve', 'Rebecca Ferguson', 'Timothée Chalamet']}


--- Unseen #5: Sarcastic negative (expected to fool the model into positive) ---
Text:     This movie really cured my insomnia! Thank you, director Michael Bay, for making the longest ten-minute story feel like a peaceful five-hour nap.
Expected: negative | ['Michael Bay']
Model (1.49s): {'sentiment': 'positive', 'sentiment_score': 0.85, 'entities': ['Michael Bay']}


## 8. Additional Diagnostic Experiments

- **8.1 Code Sensitivity:** Demonstrates what happens when a cleaning line is removed from `clean_json_text`.
- **8.2 Tokenization Analysis:** Inspects how the Qwen2.5 tokenizer splits a Turkish agglutinative word and compares Turkish vs English token counts.

In [11]:
# 8.1 Code Sensitivity: Impact of removing the Markdown fence cleanup step
# Pick an example repaired by clean_json_text, or a standard fenced JSON response
repaired_samples = [r for r in raw_failures if r["cleaned_json_ok"]]
sample_fenced = (
    repaired_samples[0]["raw_output"]
    if repaired_samples
    else '```json\n{"sentiment": "positive", "sentiment_score": 0.9, "entities": ["Tom Hanks"]}\n```'
)

print("Sample input:")
print(repr(sample_fenced))

print("\n1) With clean_json_text() enabled:")
cleaned_ok = clean_json_text(sample_fenced)
print("   Parsed:", json.loads(cleaned_ok))

print("\n2) With Markdown fence cleanup & brace extraction removed:")
broken_text = sample_fenced.strip()
try:
    json.loads(broken_text)
except json.JSONDecodeError as err:
    print(f"   FAILED with JSONDecodeError: {err}")

# Show that structural syntax glitches (such as duplicate '{') still require try-except
unrepaired = [r for r in raw_failures if not r["cleaned_json_ok"]]
if unrepaired:
    print("\n3) Unrepaired raw failure observed in 100-sample prompt-only run (ID", unrepaired[0]["id"], "):")
    print("   Raw:", repr(unrepaired[0]["raw_output"]))
    print("   parse_json_safe(clean_json_text(...)) ->", parse_json_safe(clean_json_text(unrepaired[0]["raw_output"])))

Sample input:
'```json\n{"sentiment": "positive", "sentiment_score": 0.9, "entities": ["Tom Hanks"]}\n```'

1) With clean_json_text() enabled:
   Parsed: {'sentiment': 'positive', 'sentiment_score': 0.9, 'entities': ['Tom Hanks']}

2) With Markdown fence cleanup & brace extraction removed:
   FAILED with JSONDecodeError: Expecting value: line 1 column 1 (char 0)

3) Unrepaired raw failure observed in 100-sample prompt-only run (ID 58 ):
   Raw: '{\n\n{\n  "sentiment": "positive",\n  "sentiment_score": 0.85,\n  "entities": [\n    "Jonathan Silver"\n  ]\n}'
   parse_json_safe(clean_json_text(...)) -> None


In [12]:
# 8.2 Turkish Tokenization Experiment using Qwen2.5 Base Tokenizer
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained("Qwen/Qwen2.5-3B-Instruct")

word = "kitaplarımızdan"
token_ids = tokenizer.encode(word)
token_pieces = [tokenizer.decode([tid]) for tid in token_ids]

print(f"Word: '{word}'")
print(f"Token count: {len(token_ids)}")
print(f"Token pieces: {token_pieces}")

en_sentence = "This movie was great and Tom Hanks gave a wonderful performance."
tr_sentence = "Bu film harikaydı ve Tom Hanks muhteşem bir performans sergiledi."

en_tokens = tokenizer.encode(en_sentence)
tr_tokens = tokenizer.encode(tr_sentence)
print(f"\nEnglish sentence ({len(en_tokens)} tokens): {[tokenizer.decode([t]) for t in en_tokens]}")
print(f"Turkish sentence ({len(tr_tokens)} tokens): {[tokenizer.decode([t]) for t in tr_tokens]}")

[transformers] PyTorch was not found. Models won't be available and only tokenizers, configuration and file/data utilities can be used.


Word: 'kitaplarımızdan'
Token count: 4
Token pieces: ['kit', 'ap', 'larımız', 'dan']

English sentence (13 tokens): ['This', ' movie', ' was', ' great', ' and', ' Tom', ' H', 'anks', ' gave', ' a', ' wonderful', ' performance', '.']
Turkish sentence (22 tokens): ['Bu', ' film', ' har', 'ik', 'ay', 'dı', ' ve', ' Tom', ' H', 'anks', ' mu', 'hte', 'ş', 'em', ' bir', ' perform', 'ans', ' ser', 'g', 'iled', 'i', '.']
